# Étape 4 – Statistiques : qu'est-ce qui influence la note client ?

**Objectif :** mesurer, avec des graphiques et des tests statistiques, quels facteurs
(délai, retard, frais de port, nombre d'articles…) sont liés à une mauvaise note.

Ce notebook lit directement les données dans la base PostgreSQL `olist` chargée par `src/etl.py`.

> 💡 Pour exécuter une cellule : clique dedans puis **Maj + Entrée**.

## 0. Préparation : librairies et connexion à la base

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine

# Permet d'importer src/config.py (qui lit le mot de passe dans .env)
sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import pg_url

engine = create_engine(pg_url())

# Style des graphiques : sobre et lisible
COULEUR = "#2a78d6"
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "font.size": 11,
})

# Test de connexion
pd.read_sql("SELECT COUNT(*) AS nb_commandes FROM orders", engine)

## 1. Comment se répartissent les notes clients ?

Avant de chercher des causes, on regarde **la forme** des données : combien d'avis à 1, 2, 3, 4 et 5 étoiles ?

In [ ]:
notes = pd.read_sql("""
    SELECT review_score AS note,
           COUNT(*)     AS nb_avis
    FROM order_reviews
    GROUP BY review_score
    ORDER BY review_score
""", engine)

notes["pct"] = (100 * notes["nb_avis"] / notes["nb_avis"].sum()).round(1)
notes

In [ ]:
fig, ax = plt.subplots()
barres = ax.bar(notes["note"], notes["pct"], color=COULEUR, width=0.6)

# Valeur affichée au-dessus de chaque barre
ax.bar_label(barres, labels=[f"{p} %" for p in notes["pct"]], padding=3)

ax.set_title("Répartition des notes clients", loc="left", fontweight="bold")
ax.set_xlabel("Note (étoiles)")
ax.set_ylabel("% des avis")
ax.set_xticks([1, 2, 3, 4, 5])
plt.show()

### 🎯 À toi d'interpréter

1. Quelle note est **la plus fréquente** ? Et la **deuxième** ?
2. La forme du graphique est-elle « en cloche » (beaucoup de 3, peu de 1 et de 5) ? Sinon, à quoi ressemble-t-elle ?
3. Quel **% d'avis négatifs** (1 ou 2 étoiles) ?

✍️ *Écris ta réponse ici (double-clique sur cette cellule pour la modifier) :*

…